# Baseline Modeling and Error Analysis

The objective of this notebook is to:
- Build baseline models,
- Compare their results,
- Analyze false negatives,
- Analyze false positives,
- Formulate hypotheses for further experiments.

## Environment and imports

Import libraries and evaluation metrics.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

## Helper functions

Define model evaluation and error-analysis helpers.

In [2]:
DECISION_THRESHOLD = 0.5

def predict_with_threshold(model, X, threshold=DECISION_THRESHOLD):
    """Return probabilities and class predictions from one shared calculation."""
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    return y_pred, y_proba


def get_metrics(name, y_true, y_pred, y_proba, threshold=DECISION_THRESHOLD):
    return {
        "model": name,
        "decision_threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
    }


def evaluate_model(name, y_true, y_pred, y_proba, threshold=DECISION_THRESHOLD):
    metrics = get_metrics(name, y_true, y_pred, y_proba, threshold)
    print(name)
    print("Decision Threshold:", threshold)
    for metric in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
        print(f"{metric.upper()}: {metrics[metric]:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("Classification Report:\n", classification_report(y_true, y_pred, zero_division=0))


def get_false_negatives(debug_df, model_name):
    return debug_df[
        (debug_df["y_true"] == 1)
        & (debug_df[f"{model_name}_pred"] == 0)
    ]


def add_failure_mechanism_flags(df):
    """Add the same diagnostic rules to every error-analysis table."""
    df = df.copy()
    df["OSF_distance"] = (
        df["Tool wear [min]"] * df["Torque [Nm]"] - 11003.2
    )
    df["OSF_like"] = df["OSF_distance"] >= 0
    df["HDF_like"] = (
        df["Temperature difference"].between(7.6, 8.6)
        & df["Rotational speed [rpm]"].between(1212, 1379)
    )
    df["PWF_like"] = (
        df["Power [W]"].between(1148.44, 3477.24)
        | df["Power [W]"].between(9004.43, 10469.92)
    )
    df["TWF_like"] = df["Tool wear [min]"].between(198, 253)
    df["n_like_mechanisms"] = df[
        ["OSF_like", "HDF_like", "PWF_like", "TWF_like"]
    ].sum(axis=1)
    return df


## Load prepared data

Load scaled features and original-index metadata.

In [3]:
# Dane skalowane — do trenowania modeli
X_train = pd.read_csv(
    "../data/processed/X_train_scaled.csv",
    index_col="record_index"
)

X_test = pd.read_csv(
    "../data/processed/X_test_scaled.csv",
    index_col="record_index"
)

# Dane nieskalowane — do interpretacji rekordów i błędów modeli
X_train_raw = pd.read_csv(
    "../data/processed/X_train_raw.csv",
    index_col="record_index"
)

X_test_raw = pd.read_csv(
    "../data/processed/X_test_raw.csv",
    index_col="record_index"
)

# Target
y_train = pd.read_csv(
    "../data/processed/y_train.csv",
    index_col="record_index"
).squeeze("columns")

y_test = pd.read_csv(
    "../data/processed/y_test.csv",
    index_col="record_index"
).squeeze("columns")

### Validate loaded data

Check shapes, indexes and class distributions.

In [4]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

print(X_train.head())
print(y_train.value_counts())
print(y_test.value_counts())

(8000, 8)
(2000, 8)
(8000,)
(2000,)
              Type_L  Type_M  Process temperature [K]  Temperature difference  \
record_index                                                                    
3693               1       0                 0.874402               -0.800498   
590                1       0                -0.408876                1.900381   
6770               0       1                 0.536697               -0.500400   
1412               0       1                -0.071171                1.100121   
3298               1       0                 0.198993               -0.900531   

              Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  
record_index                                                                   
3693                       -0.100148    -0.851496  -1.182391         1.485820  
590                        -0.122496    -0.329786  -0.421646         1.973739  
6770                       -0.167192     0.011332   0.052397        -1.30004

## Baseline models

Train simple reference and tree-based classifiers.

### Dummy classifier

Establish a majority-class reference score.

In [5]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)


,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.97,0.03]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](8,)","['Type_L','Type_M','Process temperature [K]',...,'Torque [Nm]','Power [W]', 'Tool wear [min]']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,8
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


Observation: The dummy model has high accuracy but detects no failures. It is not useful for this task.



### Logistic regression

Train a linear baseline with balanced classes.

In [6]:
log_reg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
)
log_reg.fit(X_train, y_train)


,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

Observation: Logistic regression detects failures, but its low precision produces many false alarms.



### Random forest

Train a nonlinear ensemble baseline.

In [7]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
)
random_forest.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.

Observation: Random Forest gives the best precision/recall balance. The exact error counts are displayed by the evaluation code.

### Gradient boosting

Train a sequential tree-based baseline.

In [8]:
gradient_boosting = GradientBoostingClassifier(random_state=42)
gradient_boosting.fit(X_train, y_train)


,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (im

Observation: Gradient Boosting produces fewer false positives, but misses more failures than Random Forest. The exact error counts are displayed by the evaluation code.

### Compare baseline models

Collect and compare classification metrics.

In [9]:
predictions = {}
for name, model in {
    "dummy_classifier": dummy,
    "logistic_regression": log_reg,
    "random_forest": random_forest,
    "gradient_boosting": gradient_boosting,
}.items():
    y_pred, y_proba = predict_with_threshold(model, X_test)
    predictions[name] = {"pred": y_pred, "proba": y_proba}

evaluate_model("Dummy Classifier", y_test, predictions["dummy_classifier"]["pred"], predictions["dummy_classifier"]["proba"])
evaluate_model("Logistic Regression", y_test, predictions["logistic_regression"]["pred"], predictions["logistic_regression"]["proba"])
evaluate_model("Random Forest", y_test, predictions["random_forest"]["pred"], predictions["random_forest"]["proba"])
evaluate_model("Gradient Boosting", y_test, predictions["gradient_boosting"]["pred"], predictions["gradient_boosting"]["proba"])

results_df = pd.DataFrame([
    get_metrics(
        name.replace("_", " ").title(),
        y_test,
        values["pred"],
        values["proba"],
    )
    for name, values in predictions.items()
])
print(results_df.round(4))


Dummy Classifier
Decision Threshold: 0.5
ACCURACY: 0.9670
PRECISION: 0.0000
RECALL: 0.0000
F1: 0.0000
ROC_AUC: 0.5000
PR_AUC: 0.0330
Confusion Matrix:
 [[1934    0]
 [  66    0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98      1934
           1       0.00      0.00      0.00        66

    accuracy                           0.97      2000
   macro avg       0.48      0.50      0.49      2000
weighted avg       0.94      0.97      0.95      2000

Logistic Regression
Decision Threshold: 0.5
ACCURACY: 0.8475
PRECISION: 0.1595
RECALL: 0.8485
F1: 0.2686
ROC_AUC: 0.9202
PR_AUC: 0.4192
Confusion Matrix:
 [[1639  295]
 [  10   56]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.85      0.91      1934
           1       0.16      0.85      0.27        66

    accuracy                           0.85      2000
   macro avg       0.58      0.85      0.59   

## False-negative analysis

Identify failures that were not detected by the models.

## Observations from model comparison

- Random Forest provides the best precision/recall balance.
- Gradient Boosting achieves the best ROC-AUC and PR-AUC.
- Accuracy alone is insufficient for this imbalanced target.
- Further changes should begin with cross-validation.

In [10]:
debug_df = X_test_raw.copy()
debug_df["y_true"] = y_test

for name, values in predictions.items():
    if name == "dummy_classifier":
        continue
    debug_df[f"{name}_proba"] = values["proba"]
    debug_df[f"{name}_pred"] = values["pred"]

print(debug_df.head())
print(debug_df.shape)


              Type_L  Type_M  Process temperature [K]  Temperature difference  \
record_index                                                                    
7566               1       0                    311.0                    10.7   
8199               1       0                    310.7                    11.5   
9316               0       0                    309.4                    10.9   
2882               1       0                    309.6                     9.0   
4293               1       0                    310.0                     8.2   

              Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  \
record_index                                                                    
7566                            1613         34.7    5861.28              142   
8199                            1737         27.0    4911.25              225   
9316                            1417         41.2    6113.58              155   
2882                       

In [11]:
model_names = ["logistic_regression", "random_forest", "gradient_boosting"]
fn_summary = pd.DataFrame({
    "model": model_names,
    "false_negatives": [
        len(get_false_negatives(debug_df, name)) for name in model_names
    ],
})
print(fn_summary)


                 model  false_negatives
0  logistic_regression               10
1        random_forest               11
2    gradient_boosting               14


In [12]:
print(get_false_negatives(debug_df, 'logistic_regression'))


              Type_L  Type_M  Process temperature [K]  Temperature difference  \
record_index                                                                    
8199               1       0                    310.7                    11.5   
4151               0       1                    310.4                     8.5   
4475               1       0                    310.5                     7.8   
4034               1       0                    310.8                     8.8   
4833               1       0                    311.9                     8.5   
8609               1       0                    308.3                    10.9   
9018               1       0                    308.1                    10.8   
442                1       0                    308.5                    11.1   
3787               1       0                    310.8                     8.5   
7510               1       0                    311.8                    11.3   

              Rotational sp

In [13]:
print(get_false_negatives(debug_df, 'random_forest'))


              Type_L  Type_M  Process temperature [K]  Temperature difference  \
record_index                                                                    
8199               1       0                    310.7                    11.5   
4475               1       0                    310.5                     7.8   
4034               1       0                    310.8                     8.8   
4833               1       0                    311.9                     8.5   
2941               0       1                    309.6                     8.9   
8609               1       0                    308.3                    10.9   
8026               1       0                    311.9                    11.2   
9663               1       0                    310.1                    11.0   
9758               1       0                    309.8                    11.2   
9018               1       0                    308.1                    10.8   
7510               1       0

In [14]:
print(get_false_negatives(debug_df, 'gradient_boosting'))


              Type_L  Type_M  Process temperature [K]  Temperature difference  \
record_index                                                                    
8199               1       0                    310.7                    11.5   
4034               1       0                    310.8                     8.8   
2494               1       0                    308.8                     9.7   
2941               0       1                    309.6                     8.9   
1085               1       0                    307.8                    10.8   
8609               1       0                    308.3                    10.9   
8026               1       0                    311.9                    11.2   
9653               1       0                    309.9                    10.9   
3760               1       0                    310.9                     8.6   
9663               1       0                    310.1                    11.0   
9758               1       0

In [15]:
model_names = ["logistic_regression", "random_forest", "gradient_boosting"]
fn_summary = pd.DataFrame({
    "model": model_names,
    "false_negatives": [
        len(get_false_negatives(debug_df, name)) for name in model_names
    ],
})
print(fn_summary)

                 model  false_negatives
0  logistic_regression               10
1        random_forest               11
2    gradient_boosting               14


In [16]:
fn_indices = {
    name: set(get_false_negatives(debug_df, name).index)
    for name in ["logistic_regression", "random_forest", "gradient_boosting"]
}
for name, indices in fn_indices.items():
    print(f"{name} FN:", len(indices))

logistic_regression FN:

 10
random_forest FN: 11
gradient_boosting FN: 14


In [17]:
fn_all_index = set.intersection(*fn_indices.values())
print("Failures missed by all models:", len(fn_all_index))
print(fn_all_index)

for model_name in ["logistic_regression", "random_forest", "gradient_boosting"]:
    debug_df[f"missed_by_{model_name}"] = (
        (debug_df["y_true"] == 1) & (debug_df[f"{model_name}_pred"] == 0)
    )
debug_df["missed_by_n_models"] = debug_df[[
    "missed_by_logistic_regression", "missed_by_random_forest",
    "missed_by_gradient_boosting"
]].sum(axis=1)


Failures missed by all models: 5
{8609, 4034, 8199, 7510, 9018}


In [18]:
hard_cases = debug_df[
    (debug_df["y_true"] == 1) &
    (debug_df["missed_by_n_models"] > 0)
].copy()
diagnostic_cols = [
    "y_true",
    "logistic_regression_pred",
    "logistic_regression_proba",
    "random_forest_pred",
    "random_forest_proba",
    "gradient_boosting_pred",
    "gradient_boosting_proba",
    "Process temperature [K]",
    "Temperature difference",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Power [W]",
    "Tool wear [min]",
    "Type_L",
    "Type_M"
]

print(
    hard_cases[diagnostic_cols + ["missed_by_n_models"]]
    .sort_values("missed_by_n_models", ascending=False)
)

              y_true  logistic_regression_pred  logistic_regression_proba  \
record_index                                                                
8199               1                         0                   0.070138   
8609               1                         0                   0.249070   
4034               1                         0                   0.431925   
9018               1                         0                   0.132736   
7510               1                         0                   0.137707   
9758               1                         1                   0.778214   
4475               1                         0                   0.163843   
8026               1                         1                   0.635897   
9663               1                         1                   0.590242   
2941               1                         1                   0.605943   
4833               1                         0                   0.131180   

In [19]:
overlap_summary = pd.DataFrame({
    "comparison": [
        "Logistic Regression / Random Forest",
        "Logistic Regression / Gradient Boosting",
        "Random Forest / Gradient Boosting",
        "All three models",
    ],
    "shared_false_negatives": [
        len(fn_indices["logistic_regression"] & fn_indices["random_forest"]),
        len(fn_indices["logistic_regression"] & fn_indices["gradient_boosting"]),
        len(fn_indices["random_forest"] & fn_indices["gradient_boosting"]),
        len(fn_all_index),
    ],
})
print(overlap_summary)


                                comparison  shared_false_negatives
0      Logistic Regression / Random Forest                       7
1  Logistic Regression / Gradient Boosting                       5
2        Random Forest / Gradient Boosting                       9
3                         All three models                       5


### Inspect common false negatives

Review failures missed by all baseline models.

In [20]:
fn_every = sorted(fn_all_index)

print(f"Common false negatives: {len(fn_every)}")
print(fn_every)

Common false negatives: 5
[4034, 7510, 8199, 8609, 9018]


In [21]:
suspect_cases = X_test_raw.loc[sorted(fn_all_index)].copy()
suspect_cases["y_true"] = y_test.loc[suspect_cases.index]
print(suspect_cases)


              Type_L  Type_M  Process temperature [K]  Temperature difference  \
record_index                                                                    
4034               1       0                    310.8                     8.8   
7510               1       0                    311.8                    11.3   
8199               1       0                    310.7                    11.5   
8609               1       0                    308.3                    10.9   
9018               1       0                    308.1                    10.8   

              Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  \
record_index                                                                    
4034                            1615         29.0    4904.55              235   
7510                            1524         38.9    6208.16              214   
8199                            1737         27.0    4911.25              225   
8609                       

### Interpret false-negative patterns

Group missed failures by suspected mechanisms and boundary cases.

442 - very close to the lower PWF boundary

3760, 3787, 3829, 4151, 4833, 4475 - close to the HDF boundaries for rotational speed and temperature difference

1085, 2494, 9653, 9663, 8026 - the models did not include the OSF criterion

2941, 9758 - high rotational speed combined with high tool wear [TWF]; EDA did not show a clear relationship between rotational speed and TWF

7510, 9018, 4034, 8609, 8199 - high tool wear [TWF]


3787, 442, 3760, 3829, 1085, 2494, 9653, 4151 - missed by one model

4833, 2941, 9663, 8026, 4475, 9758 - missed by two models

7510, 9018, 4034, 8609, 8199 - missed by all three models

## False-positive analysis

Inspect normal records incorrectly flagged as failures.

In [22]:
fp_rf = debug_df[(debug_df["y_true"] == 0) & (debug_df["random_forest_pred"] == 1)].copy()
fp_gb = debug_df[(debug_df["y_true"] == 0) & (debug_df["gradient_boosting_pred"] == 1)].copy()
print("False positives - Random Forest:", len(fp_rf))
print("False positives - Gradient Boosting:", len(fp_gb))


False positives - Random Forest: 10
False positives - Gradient Boosting: 7


In [23]:
print(fp_rf)
print(fp_gb)


              Type_L  Type_M  Process temperature [K]  Temperature difference  \
record_index                                                                    
3764               1       0                    311.0                     8.6   
4898               0       0                    312.4                     8.8   
8198               0       1                    310.7                    11.4   
7081               1       0                    310.4                     9.7   
7677               1       0                    311.7                    11.1   
5653               1       0                    311.7                     9.3   
4115               0       0                    310.6                     8.6   
4110               1       0                    310.6                     8.6   
7255               0       1                    310.3                    10.1   
4862               1       0                    312.1                     8.6   

              Rotational sp

In [24]:
print("RF - y_true values:")
print(fp_rf["y_true"].value_counts())

print("\nRF - prediction values:")
print(fp_rf["random_forest_pred"].value_counts())

print("\nGB - y_true values:")
print(fp_gb["y_true"].value_counts())

print("\nGB - prediction values:")
print(fp_gb["gradient_boosting_pred"].value_counts())

RF - y_true values:
y_true
0    10
Name: count, dtype: int64

RF - prediction values:
random_forest_pred
1    10
Name: count, dtype: int64

GB - y_true values:
y_true
0    7
Name: count, dtype: int64

GB - prediction values:
gradient_boosting_pred
1    7
Name: count, dtype: int64


In [25]:
fp_rf_summary = fp_rf.drop(
    columns=["logistic_regression_pred", "logistic_regression_proba", "gradient_boosting_pred",
             "gradient_boosting_proba", "missed_by_logistic_regression",
             "missed_by_gradient_boosting", "missed_by_n_models",
             "missed_by_random_forest"],
).copy()


### Random Forest false positives

List and inspect normal records flagged by Random Forest.

False positives are listed in the following output.
4898
8198
7081
7677
4115
4110
7255
4862

In [26]:
print(fp_rf_summary["random_forest_proba"].min())
print(fp_rf_summary["random_forest_proba"].max())

0.5
0.75


### Random Forest probability range

Check how close the false positives are to the decision threshold.

The probability range is calculated by the code above.

In [27]:
fp_rf_summary = fp_rf.drop(
    columns=["logistic_regression_pred", "logistic_regression_proba", "gradient_boosting_pred",
             "gradient_boosting_proba", "missed_by_logistic_regression",
             "missed_by_gradient_boosting", "missed_by_n_models",
             "missed_by_random_forest"],
).copy()


### Gradient Boosting false positives

List and inspect normal records flagged by Gradient Boosting.

False positives are listed in the following output.

In [28]:
fp_gb_summary = fp_gb.drop(columns=["logistic_regression_pred", "logistic_regression_proba", "random_forest_pred", "random_forest_proba", "missed_by_logistic_regression", "missed_by_gradient_boosting", "missed_by_n_models", "missed_by_random_forest"]).copy()
print(fp_gb_summary["gradient_boosting_proba"].min().round(2))
print(fp_gb_summary["gradient_boosting_proba"].max().round(2))

0.54
0.85


### Gradient Boosting probability range

Check the confidence range of false-positive predictions.

The probability range is calculated by the code above.

In [29]:
fp_gb_summary = fp_gb.drop(
    columns=["logistic_regression_pred", "logistic_regression_proba", "random_forest_pred",
             "random_forest_proba", "missed_by_logistic_regression",
             "missed_by_gradient_boosting", "missed_by_n_models",
             "missed_by_random_forest"],
).copy()


In [30]:
for frame, model_name in [
    (fp_rf_summary, "random_forest"),
    (fp_gb_summary, "gradient_boosting"),
]:
    frame["margin"] = frame[f"{model_name}_proba"] - DECISION_THRESHOLD

fp_rf_summary = add_failure_mechanism_flags(fp_rf_summary)
fp_gb_summary = add_failure_mechanism_flags(fp_gb_summary)


In [31]:
for frame, model_name in [
    (fp_rf_summary, "random_forest"),
    (fp_gb_summary, "gradient_boosting"),
]:
    frame["margin"] = frame[f"{model_name}_proba"] - DECISION_THRESHOLD

fp_rf_summary = add_failure_mechanism_flags(fp_rf_summary)
fp_gb_summary = add_failure_mechanism_flags(fp_gb_summary)


In [32]:
fp_gb_summary["n_like_mechanisms"]


record_index
6925    2
3764    1
7677    1
1507    2
7935    1
4862    1
9671    1
Name: n_like_mechanisms, dtype: int64

## Export baseline results

Save metrics and error-analysis tables.

In [33]:
results_path = Path("../results")
results_path.mkdir(parents=True, exist_ok=True)

# Explicitly report records sitting exactly on the decision boundary;
# the ">= threshold" convention classifies them as class 1.
for name, values in predictions.items():
    if name == "dummy_classifier":
        continue
    tied_records = debug_df.index[values["proba"] == DECISION_THRESHOLD].tolist()
    print(
        f"{name}: {len(tied_records)} record(s) with probability == "
        f"{DECISION_THRESHOLD}", tied_records if tied_records else ""
    )

results_df.round(6).to_csv(results_path / "model_results_baseline.csv", index=False)

# Report copies round probabilities for readability;
# in-memory frames keep full precision for computations.
PROBA_DECIMALS = 4

false_negatives_export = hard_cases[diagnostic_cols + ["missed_by_n_models"]].sort_values("missed_by_n_models", ascending=False)
proba_columns = [column for column in false_negatives_export.columns if column.endswith("_proba")]
false_negatives_export[proba_columns] = false_negatives_export[proba_columns].round(PROBA_DECIMALS)
false_negatives_export.to_csv(results_path / "false_negatives_baseline.csv", index=True, index_label="record_index")

fp_rf_summary["model"] = "Random Forest"
fp_gb_summary["model"] = "Gradient Boosting"
fp_rf_summary = fp_rf_summary.rename(columns={"random_forest_proba": "predicted_probability"})
fp_gb_summary = fp_gb_summary.rename(columns={"gradient_boosting_proba": "predicted_probability"})
raw_feature_columns = [
    "Process temperature [K]", "Temperature difference", "Rotational speed [rpm]",
    "Torque [Nm]", "Power [W]", "Tool wear [min]", "Type_L", "Type_M",
]
false_positive_columns = [
    "model", "predicted_probability", "margin", "OSF_distance",
    "OSF_like", "HDF_like", "PWF_like", "TWF_like", "n_like_mechanisms",
    *raw_feature_columns,
]
false_positives_export = (
    pd.concat([fp_rf_summary[false_positive_columns], fp_gb_summary[false_positive_columns]])
    .reset_index(names="record_index")
)
false_positives_export[["predicted_probability", "margin", "OSF_distance"]] = (
    false_positives_export[["predicted_probability", "margin", "OSF_distance"]].round(PROBA_DECIMALS)
)
false_positives_export.to_csv(results_path / "false_positives_baseline.csv", index=False)
print("Saved files to:", results_path)


logistic_regression: 0 record(s) with probability == 0.5 
random_forest: 1 record(s) with probability == 0.5 [5653]
gradient_boosting: 0 record(s) with probability == 0.5 
Saved files to: ..\results
